# Exercise 2.3: Cleaning, Missing Values & Duplicates (Angola IEA)

This notebook reads the typed checkpoint from 2.2 and turns it into a defensible
cleaned dataset.

You will practice:
- Working on a copy, never on the loaded data
- Telling missing by design apart from missing by error
- Recoding sentinel codes that pandas cannot see are missing
- Resolving duplicates on a compound key
- Writing validation rules that actually remove something
- Catching a cross column problem that `info()` and `describe()` cannot see

> **Pipeline:** run Exercise 2.2 first. Reads and overwrites `10_cleaned/`.

### Path Setup (run first)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
typed_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_typed.csv')

STR_COLS = {
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
}
df = pd.read_csv(typed_path, dtype=STR_COLS)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded typed checkpoint:', df.shape)
df.head()

---

## Task 1: Never destroy the input

Every change goes on a copy. The loaded frame stays untouched so you can compare
before and after at any point, and restart when an assumption turns out wrong.

In [ ]:
df_clean = df.  # your code here
print('Working copy:', df_clean.shape)

---

## Task 2: Detect missing values

Count them, express them as a share, and look at the shape of the problem before
deciding anything.

In [ ]:
missing = pd.DataFrame({
    'n_missing': df_clean.isna().sum(),
    'pct_missing': (df_clean.isna().mean() * 100).round(1),
})
missing.sort_values('pct_missing', ascending=False)

In [ ]:
counts = df_clean.isna().sum()
counts[counts > 0].sort_values().plot(kind='barh', color='coral', figsize=(9, 7))
plt.title('Missing values by column')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

**Questions:**

- How many columns are 100% missing? What range does the rest of the missing
  columns fall into?
- What does the roughly 78% missing group represent, and why is `available_2wk`
  missing in 98.5% of rows?
- What two groups does the bar chart make obvious?

---

## Task 3: Drop the columns that hold nothing

`hh_size_reported` and `hh_adults_reported` should hold household size and the
number of adults. In this file they are empty in all 53,353 rows.

In [ ]:
empty_cols =   # your code here: columns where isna().all()
print('Entirely empty columns:', empty_cols)

print('Before:', df_clean.shape)
df_clean = df_clean.drop(columns=empty_cols)
print('After: ', df_clean.shape)

**Questions:**

- How many columns are entirely empty, and how many remain after dropping them?
  Why detect them with `isna().all()` instead of naming them directly?
- Is household size a useless piece of information? What happens to it later in
  the series?
- What does the Q3 2025 file tell you about trusting the newest extract by
  default?

---

## Task 4: Coded missing values

pandas only recognises blanks and `NaN`. Survey files also use sentinel codes
that look like ordinary numbers and silently corrupt every statistic they touch.

| Column | Sentinel | Meaning |
|---|---|---|
| `hours_usual`, `hours_actual` | 997 | NAO SABE, does not know |
| `job_start_year` | 9997 | NAO SABE, does not know |

In [ ]:
def recode_coded_missing(series, codes):
    """Replace coded missing values with NaN."""
    # your code here: return series.replace(codes, np.nan)
    return


print('job_start_year mean BEFORE:', round(df_clean['job_start_year'].mean(), 1))
print('hours_usual max BEFORE:    ', df_clean['hours_usual'].max())

df_clean['hours_usual'] = recode_coded_missing(df_clean['hours_usual'], [997, 998, 999])
df_clean['hours_actual'] = recode_coded_missing(df_clean['hours_actual'], [997, 998, 999])
df_clean['job_start_year'] = recode_coded_missing(df_clean['job_start_year'], [9997, 9998, 9999])

print('job_start_year mean AFTER: ', round(df_clean['job_start_year'].mean(), 1))
print('hours_usual max AFTER:     ', df_clean['hours_usual'].max())

**Questions:**

- What does the mean of `job_start_year` look like before and after recoding the
  sentinel codes? What does the "before" number actually represent?
- What happens to the maximum of `hours_usual` after recoding? Is the result
  fully plausible?
- Why should you never recode a value of 0 as missing without checking the
  questionnaire documentation first?

---

## Task 5: Missing by design is not missing by error

The textbook first move is `dropna()`. On a survey with skip patterns it is a
catastrophe. Measure it before you trust it.

In [ ]:
print('Rows now:                   ', len(df_clean))
print('Rows if we called dropna(): ', len(df_clean.dropna()))

In [ ]:
# The right rule: only the identifiers are non negotiable
print('Before:', df_clean.shape)
df_clean = df_clean.dropna(  # your code here: subset=['household_id', 'person_no'] )
print('After: ', df_clean.shape)

**Questions:**

- How many rows survive a call to `df_clean.dropna()` with no arguments? Why is
  the result so extreme on a survey with skip patterns?
- How many rows does dropping only on `household_id` and `person_no` remove?
  Why is it still worth running even when it removes nothing?
- Why would filling the empty labour force columns with a median be worse than
  leaving them empty?

---

## Task 6: Duplicates on a compound key

No two rows here are identical, so `duplicated()` alone finds nothing. The real
key is the pair `household_id` plus `person_no`: one row per person per household.

In [ ]:
print('Exact duplicate rows:', df_clean.duplicated().sum())

dup_mask = df_clean.duplicated(  # your code here: subset and keep=False )
print('Rows sharing a person key:', dup_mask.sum())
df_clean[dup_mask].sort_values(['household_id', 'person_no'])[
    ['household_id', 'person_no', 'age', 'sex', 'rel_to_head', 'hours_usual']]

In [ ]:
# Keep the most complete record in each group
df_clean['missing_count'] = df_clean.isna().sum(axis=1)

print('Before:', df_clean.shape)
df_clean = (
    df_clean
    .sort_values(['household_id', 'person_no', 'missing_count'])
    .drop_duplicates(  # your code here: subset and keep='first' )
    .drop(columns='missing_count')
)
print('After: ', df_clean.shape)

**Questions:**

- How many exact duplicate rows are there, and how many rows share a
  `household_id` plus `person_no` key? Why does only the compound key find them?
- How many rows are removed once the duplicates are resolved, and what is the
  resulting row count?
- Why use `keep=False` when inspecting duplicates instead of the default
  `keep='first'`?

---

## Task 7: Validation rules

Some values are not missing, they are impossible. A rule that removes nothing is
not a rule, so check the count each time.

In [ ]:
print('age above 100:', (df_clean['age'] > 100).sum())
print('Before:', df_clean.shape)
df_clean = df_clean[df_clean['age'].  # your code here: between(0, 100) ]
print('After: ', df_clean.shape)

In [ ]:
df_clean.boxplot(column='hours_usual', figsize=(6, 5))
plt.title('Usual weekly hours, after sentinel recode')
plt.tight_layout()
plt.show()

In [ ]:
# 98 hours is 14 hours a day, every day. Keep the missing values.
print('Before:', df_clean.shape)
hours_ok = df_clean['hours_usual'].isna() | # your code here: between(0, 98)
df_clean = df_clean[hours_ok]
print('After: ', df_clean.shape)

**Questions:**

- How many people does the age rule remove, and what ages do they have?
- How many rows does the hours rule remove? What would happen if the
  `isna() |` guard were left out?
- Why would a bound of 120 hours have been a useless rule on this data?

---

## Task 8: A rule that `info()` and `describe()` cannot catch

Every household should have exactly one head, `rel_to_head == 1`. No summary
statistic will tell you whether that holds, because it is a relationship between
rows.

In [ ]:
heads = df_clean[df_clean['rel_to_head'] == 1]
heads_per_household = heads['household_id'].  # your code here

print('Households with two or more heads:', (heads_per_household > 1).sum())
print('Households with no head recorded: ',
      df_clean['household_id'].nunique() - heads_per_household.index.nunique())

**Questions:**

- After cleaning, how many households have two or more heads, and how many have
  none at all?
- How many households had two heads before the Task 6 deduplication? What does
  that tell you about the cause of the problem?
- Why do `info()` and `describe()` never catch this kind of problem?

---

## Task 9: Save the cleaned dataset

Overwrite nothing in `0_raw/`. Write the result to `10_cleaned/` and reload it to
confirm it survives the round trip.

In [ ]:
df_clean = df_clean.reset_index(drop=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

df_clean.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', df_clean.shape)

In [ ]:
check = pd.read_csv(out_path, dtype=STR_COLS)
print('Reloaded:', check.shape)
print()
print('Rows removed in total:', 53353 - len(check))
check.head()

**Questions:**

- What is the final shape of the cleaned file, and how does the row count break
  down against the 53,353 starting rows?
- Why does making every removal countable matter for defending the cleaning to
  someone else?
- What share of rows were removed in total? Compare that to the effect of the
  sentinel recoding on the results.